# Worked CNN Example

Nguyễn Ngọc Hoàng Nam - B23DCCN585 | Assignment 04

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Hãy mở notebook từ thư mục repository.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
from src.data import DATASETS, load_data, data_root
print('Python:', sys.version.split()[0])
print('Dữ liệu:', data_root())

Python: 3.10.20
Dữ liệu: E:\PTHTTM\ASG_04_data


## 1. Ví dụ ảnh nhân tạo xuyên suốt

Ảnh 5×5, kernel 2×2, ReLU, MaxPool 2×2, Flatten, Dense 3 lớp, cross-entropy và một bước Adam. Đây là minh họa số học, không phải thí nghiệm trên MNIST/CIFAR. Mã bên dưới chính là module tạo các giá trị được đưa vào báo cáo.

In [2]:
"""Executable synthetic example: image -> Conv -> ReLU -> Pool -> Dense -> CE."""
import json
from pathlib import Path
import numpy as np
from src.numpy_cnn import Conv2D, ReLU, MaxPool2D, Flatten, Dense, Adam, cross_entropy
from src.data import ROOT

def example():
    rng=np.random.default_rng(0)
    x=np.array([[1,0,2,3,1],[4,6,6,8,2],[3,1,1,0,2],
                [1,2,2,4,0],[0,1,3,1,2]],dtype='float32')[None,None]
    conv=Conv2D(1,1,rng,kernel=2,padding=0)
    conv.w[:]=[[[[1,0],[0,-1]]]];conv.b[:]=0
    dense=Dense(4,3,rng)
    dense.w[:]=[[.1,-.2,.0],[.0,.1,-.1],[-.1,.0,.2],[.2,-.1,.1]]
    dense.b[:]=[0,.1,-.1]
    layers=[conv,ReLU(),MaxPool2D(),Flatten(),dense]
    record={'purpose':'synthetic arithmetic illustration, not a dataset experiment',
            'image':x[0,0].tolist(),'kernel':conv.w[0,0].tolist(),
            'dense_weights':dense.w.tolist(),'dense_bias':dense.b.tolist(),'label':1}
    z=x
    for name,layer in zip(['convolution','relu','pool','flatten','logits'],layers):
        z=layer.forward(z,True);record[name]=z.squeeze().tolist()
    loss,g=cross_entropy(z,np.array([1]))
    record['loss']=loss;record['logit_gradient']=g[0].tolist()
    prob=np.exp(z-z.max());prob/=prob.sum();record['probabilities']=prob[0].tolist()
    for layer in reversed(layers):g=layer.backward(g)
    record['input_gradient']=g[0,0].tolist();record['kernel_gradient']=conv.dw[0,0].tolist()
    record['dense_gradient']=dense.dw.tolist()
    before=conv.w.copy();Adam(lr=.001).step([p for layer in layers for p in layer.params()])
    record['kernel_after_adam']=conv.w[0,0].tolist()
    assert np.isfinite(loss) and np.allclose(prob.sum(),1)
    # Independent direct loop verifies the vectorized convolution.
    direct=np.array([[float((x[0,0,i:i+2,j:j+2]*before[0,0]).sum())
                      for j in range(4)] for i in range(4)])
    np.testing.assert_allclose(direct,record['convolution'])
    return record



In [3]:
actual=example()
reference=json.loads((ROOT/'results/teaching_example.json').read_text())
for key,value in actual.items():
    if key!='purpose':np.testing.assert_allclose(value,reference[key],atol=1e-6,rtol=1e-6)
print('Toàn bộ giá trị khớp file dùng cho báo cáo.')

Toàn bộ giá trị khớp file dùng cho báo cáo.


## 2. Convolution, phi tuyến và giảm không gian

In [4]:
for key in ['image','kernel','convolution','relu','pool','flatten']:
    print(key);print(np.array(actual[key]))

image
[[1. 0. 2. 3. 1.]
 [4. 6. 6. 8. 2.]
 [3. 1. 1. 0. 2.]
 [1. 2. 2. 4. 0.]
 [0. 1. 3. 1. 2.]]
kernel
[[ 1.  0.]
 [ 0. -1.]]
convolution
[[-5. -6. -6.  1.]
 [ 3.  5.  6.  6.]
 [ 1. -1. -3.  0.]
 [ 0. -1.  1.  2.]]
relu
[[0. 0. 0. 1.]
 [3. 5. 6. 6.]
 [1. 0. 0. 0.]
 [0. 0. 1. 2.]]
pool
[[5. 6.]
 [1. 2.]]
flatten
[5. 6. 1. 2.]


## 3. Logits, xác suất và loss

In [5]:
for key in ['dense_weights','dense_bias','logits','probabilities','loss','logit_gradient']:
    print(key);print(np.array(actual[key]))
assert np.isclose(sum(actual['probabilities']),1)
assert np.isclose(sum(actual['logit_gradient']),0,atol=1e-6)

dense_weights
[[ 0.1 -0.2  0. ]
 [ 0.   0.1 -0.1]
 [-0.1  0.   0.2]
 [ 0.2 -0.1  0.1]]
dense_bias
[ 0.   0.1 -0.1]
logits
[ 0.80000001 -0.49999997 -0.30000001]
probabilities
[0.62289661 0.16975914 0.20734425]
loss
1.7733746767044067
logit_gradient
[ 0.62289661 -0.83024085  0.20734426]


## 4. Gradient và cập nhật

Gradient kernel nhận tổng đóng góp từ các cửa sổ đã ảnh hưởng tới loss. Gradient input có thể bằng 0 ở vị trí bị ReLU hoặc pooling chặn. Adam cập nhật theo m/v, không đơn giản lấy learning rate nhân trực tiếp với gradient như SGD.

In [6]:
for key in ['dense_gradient','kernel_gradient','input_gradient','kernel_after_adam']:
    print(key);print(np.array(actual[key]))

dense_gradient
[[ 3.11448312 -4.15120411  1.03672135]
 [ 3.73737955 -4.98144531  1.24406552]
 [ 0.62289661 -0.83024085  0.20734426]
 [ 1.24579322 -1.66048169  0.41468853]]
kernel_gradient
[[1.59836483 0.51913804]
 [0.33209634 0.64337188]]
input_gradient
[[ 0.          0.          0.          0.          0.        ]
 [ 0.          0.22833782 -0.10375851  0.          0.        ]
 [-0.02082081  0.         -0.22833782  0.10375851  0.        ]
 [ 0.          0.02082081  0.          0.22833784  0.        ]
 [ 0.          0.          0.          0.         -0.22833784]]
kernel_after_adam
[[ 9.99000013e-01 -1.00000005e-03]
 [-1.00000005e-03 -1.00100005e+00]]


## 5. Các điểm cần giải thích khi bảo vệ

1. Tại sao convolution đầu ra là 4×4?
2. Vì sao ReLU làm một số gradient bằng 0?
3. Pooling trả gradient về đâu?
4. Vì sao gradient logits có tổng gần 0?
5. Vì sao cần cộng dồn trong col2im?
6. Một bước Adam khác một bước SGD như thế nào?

Đối chiếu chương 2 và 5 của báo cáo. Code đầy đủ của mạng huấn luyện nằm trong notebook 02; ví dụ nhỏ này chỉ hỗ trợ hiểu các phép tính.